# Hydrocephalus study — first Colab execution

This notebook checks whether the public data and code can run in a cloud environment. It runs **one logistic-regression smoke test with reduced cross-validation**. Its numbers are **not** a reproduction of the published results. Everything is installed in the temporary Colab runtime; nothing is installed on your laptop.

Run cells from top to bottom. Use a CPU runtime. The setup may take several minutes.

In [ ]:
import sys, platform
print('Colab kernel Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('The study pins packages for Python 3.11; the next cell creates Python 3.11 inside Colab.')

In [ ]:
import subprocess, sys
from pathlib import Path

repo = Path('/content/shuntclf')
env_python = Path('/content/shunt-env/bin/python')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RISCSoftware/shuntclf.git', str(repo)], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
if not env_python.exists():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/shunt-env'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(env_python), '-r', str(repo / 'requirements.txt')], check=True)
subprocess.run([str(env_python), '--version'], check=True)


## Data and code smoke test

This uses the repository's dataset loader and logistic-regression pipeline. Two outer folds and two inner folds keep the first test short. The paper used a much larger repeated validation procedure, so **do not compare these metric values directly with the paper**.

In [ ]:
from textwrap import dedent

smoke_code = dedent('''
import json, sys
from pathlib import Path
from importlib.metadata import version
from sklearn.model_selection import StratifiedKFold, cross_validate
from shuntclf.dataset import get_dataset
from shuntclf.classifier_pipelines import ClfCode, get_classifier_pipeline

df, features, target = get_dataset()
X, y = df[features], df[target]
assert len(df) == 510, f'Expected 510 patients, found {len(df)}'
assert set(y.dropna().unique()) == {0, 1}, 'Expected binary shunt target'
print('Patients:', len(df), 'Processed features:', len(features), 'Target counts:', y.value_counts().to_dict())

outer = StratifiedKFold(n_splits=2, shuffle=True, random_state=0)
inner = StratifiedKFold(n_splits=2, shuffle=True, random_state=1)
model = get_classifier_pipeline(
    ClfCode.logistic_regression, random_state=0,
    search_kwargs={'scoring': 'balanced_accuracy', 'cv': inner, 'n_jobs': 2, 'verbose': 0}
)
scores = cross_validate(
    model, X, y, cv=outer,
    scoring={'auroc': 'roc_auc', 'auprc': 'average_precision', 'f1': 'f1', 'balanced_accuracy': 'balanced_accuracy'},
    error_score='raise', n_jobs=1
)
summary = {k: float(v.mean()) for k, v in scores.items() if k.startswith('test_')}
print('Reduced-CV smoke-test metrics (NOT paper reproduction):', summary)
result = {
    'status': 'smoke_test_completed',
    'python': sys.version.split()[0],
    'versions': {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'imbalanced-learn', 'xgboost']},
    'patients': len(df), 'processed_features': len(features),
    'target_counts': {str(k): int(v) for k, v in y.value_counts().items()},
    'model': 'logistic_regression', 'outer_folds': 2, 'inner_folds': 2,
    'metrics_mean': summary,
    'interpretation': 'Environment smoke test only; not comparable with published repeated-CV metrics.'
}
Path('/content/shuntclf_colab_smoke_results.json').write_text(json.dumps(result, indent=2))
print('Saved /content/shuntclf_colab_smoke_results.json')
''')
subprocess.run([str(env_python), '-c', smoke_code], cwd=str(repo), check=True)


## Save the evidence

If the previous cell succeeds, download its small JSON result and record the repository commit printed above. If it fails, keep the error output; it is useful evidence about environment reconstruction.

In [ ]:
from google.colab import files
files.download('/content/shuntclf_colab_smoke_results.json')
